# LatentDriver Assets Setup (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_assets_colab.ipynb)

This notebook prepares the **evaluation-only** substrate for `LatentDriver`, `PlanT`, and `EasyChauffeur-PPO` on Colab with Drive-backed persistence.


## What this notebook does

1. sync this repo from GitHub
2. mount Google Drive and bind canonical asset directories
3. clone and patch the pinned LatentDriver fork
4. download the public evaluation checkpoints
5. verify that the expected checkpoint files are present

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
repo_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "repo_rev": repo_rev}, indent=2, sort_keys=True))


$ git clone --branch main https://github.com/achyutmorang/latentdriver-waymax-experiments.git /content/latentdriver-waymax-experiments
Cloning into '/content/latentdriver-waymax-experiments'...
$ /usr/bin/python3 -m pip install -e /content/latentdriver-waymax-experiments
Obtaining file:///content/latentdriver-waymax-experiments
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for latentdriver-waymax-experiments (pyproject.toml): started
  Building editable for latentdriver-waymax-experiments (pyproject.toml): finished with stat

In [2]:
from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
os.environ.pop("LATENTDRIVER_RESULTS_ROOT", None)
sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


Mounted at /content/drive
{
  "checkpoints": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/checkpoints",
  "drive_root": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments",
  "preprocessed": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/preprocessed",
  "results": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/results/runs",
  "smoke": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/smoke"
}


In [3]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


$ /usr/bin/python3 scripts/bootstrap_upstream.py
Cloning into '/content/latentdriver-waymax-experiments/external/LatentDriver'...
Note: switching to '721bd6e1f87373457b743d0e0a9606d4d0727b6f'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 721bd6e Update README.md
{
  "fork_repo_url": "https://github.com/achyutmorang/LatentDriver.git",
  "patch_path": "/content/latentdriver-waymax-experiments/patches/latentdriver_eval_contract.patch",
  "patch_state": "applied",
  "pinned_commit": "721bd6e1f87373457b743d0e0a96

In [4]:
DOWNLOAD_EVALUATION_CHECKPOINTS = True

if DOWNLOAD_EVALUATION_CHECKPOINTS:
    run([sys.executable, "scripts/download_checkpoints.py", "--evaluation-only"], cwd=REPO_DIR)
else:
    print("Skipping checkpoint download.")


$ /usr/bin/python3 scripts/download_checkpoints.py --evaluation-only
{
  "easychauffeur_ppo": {
    "expected_size_bytes": 44928697,
    "matches_expected_size": true,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/easychauffeur_policy_best.pth.tar",
    "size_bytes": 44928697
  },
  "latentdriver_t2_j3": {
    "expected_size_bytes": 122413062,
    "matches_expected_size": true,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/lantentdriver_t2_J3.ckpt",
    "size_bytes": 122413062
  },
  "latentdriver_t2_j4": {
    "expected_size_bytes": 127964626,
    "matches_expected_size": true,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/lantentdriver_t2_J4.ckpt",
    "size_bytes": 127964626
  },
  "plant": {
    "expected_size_bytes": 44758979,
    "matches_expected_size": true,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/planT.ckpt",
    "size_bytes"

In [5]:
import json

config = json.loads((REPO_DIR / "configs" / "baselines" / "latentdriver_waymax_eval.json").read_text())
checkpoints_root = REPO_DIR / config["assets"]["checkpoints_root"]
summary = {}
for name, spec in config["checkpoints"].items():
    if not spec["method"]:
        continue
    path = checkpoints_root / spec["filename"]
    summary[name] = {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
        "expected_size_bytes": spec["size_bytes"],
    }
print(json.dumps(summary, indent=2, sort_keys=True))


{
  "easychauffeur_ppo": {
    "exists": true,
    "expected_size_bytes": 44928697,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/easychauffeur_policy_best.pth.tar",
    "size_bytes": 44928697
  },
  "latentdriver_t2_j3": {
    "exists": true,
    "expected_size_bytes": 122413062,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/lantentdriver_t2_J3.ckpt",
    "size_bytes": 122413062
  },
  "latentdriver_t2_j4": {
    "exists": true,
    "expected_size_bytes": 127964626,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/lantentdriver_t2_J4.ckpt",
    "size_bytes": 127964626
  },
  "plant": {
    "exists": true,
    "expected_size_bytes": 44758979,
    "path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/planT.ckpt",
    "size_bytes": 44758979
  }
}


## Next step

After this notebook succeeds, run [`latentdriver_preprocess_val_colab.ipynb`](./latentdriver_preprocess_val_colab.ipynb) to prepare smoke or full validation artifacts.